## Step 3: Load Datasets — CIFAR-10 + CIFAR-100 + MNIST

In [ ]:
import urllib.request, tarfile, gzip, struct

# ─── CIFAR-10 ───────────────────────────────────────────
def load_cifar10_batch(filepath):
    with open(filepath, 'rb') as f:
        batch = pickle.load(f, encoding='bytes')
    imgs   = batch[b'data'].astype(np.float32) / 255.0
    labels = np.array(batch[b'labels'])
    return imgs.reshape(-1, 3, 32, 32), labels

def download_cifar10():
    url  = 'https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz'
    fname= 'cifar-10-python.tar.gz'
    if not os.path.exists(fname):
        print('📥 Downloading CIFAR-10…')
        urllib.request.urlretrieve(url, fname)
    if not os.path.exists('cifar-10-batches-py'):
        print('📦 Extracting…')
        with tarfile.open(fname,'r:gz') as tar: tar.extractall()
    print('✅ CIFAR-10 ready.')

download_cifar10()
base = 'cifar-10-batches-py'
X_parts, y_parts = [], []
for i in range(1, 6):
    Xi, yi = load_cifar10_batch(f'{base}/data_batch_{i}')
    X_parts.append(Xi); y_parts.append(yi)
X_cifar_full = np.concatenate(X_parts)
y_cifar_full = np.concatenate(y_parts)
X_cifar_test_full, y_cifar_test_full = load_cifar10_batch(f'{base}/test_batch')

# ─── CIFAR-100 (v7 NEW) ─────────────────────────────────
def load_cifar100_split(filepath):
    with open(filepath, 'rb') as f:
        batch = pickle.load(f, encoding='bytes')
    imgs   = batch[b'data'].astype(np.float32) / 255.0
    labels = np.array(batch[b'fine_labels'])   # 100 fine-grained classes
    return imgs.reshape(-1, 3, 32, 32), labels

def download_cifar100():
    url  = 'https://www.cs.toronto.edu/~kriz/cifar-100-python.tar.gz'
    fname= 'cifar-100-python.tar.gz'
    if not os.path.exists(fname):
        print('📥 Downloading CIFAR-100…')
        urllib.request.urlretrieve(url, fname)
    if not os.path.exists('cifar-100-python'):
        print('📦 Extracting…')
        with tarfile.open(fname,'r:gz') as tar: tar.extractall()
    print('✅ CIFAR-100 ready.')

download_cifar100()
base100 = 'cifar-100-python'
X_cifar100_full,      y_cifar100_full      = load_cifar100_split(f'{base100}/train')
X_cifar100_test_full, y_cifar100_test_full = load_cifar100_split(f'{base100}/test')

# ─── MNIST ──────────────────────────────────────────────
def download_mnist():
    base_url = 'https://storage.googleapis.com/cvdf-datasets/mnist/'
    files = {'train_images':'train-images-idx3-ubyte.gz',
             'train_labels':'train-labels-idx1-ubyte.gz',
             'test_images' :'t10k-images-idx3-ubyte.gz',
             'test_labels' :'t10k-labels-idx1-ubyte.gz'}
    os.makedirs('mnist', exist_ok=True)
    for k, fn in files.items():
        p = f'mnist/{fn}'
        if not os.path.exists(p):
            print(f'📥 Downloading {fn}…')
            urllib.request.urlretrieve(base_url+fn, p)
    print('✅ MNIST ready.')

def read_mnist_images(path):
    with gzip.open(path,'rb') as f:
        magic,n,r,c = struct.unpack('>IIII', f.read(16))
        return np.frombuffer(f.read(),dtype=np.uint8).reshape(n,r*c).astype(np.float32)/255.

def read_mnist_labels(path):
    with gzip.open(path,'rb') as f:
        struct.unpack('>II', f.read(8))
        return np.frombuffer(f.read(),dtype=np.uint8).astype(np.int64)

download_mnist()
X_mnist_full      = read_mnist_images('mnist/train-images-idx3-ubyte.gz')
y_mnist_full      = read_mnist_labels('mnist/train-labels-idx1-ubyte.gz')
X_mnist_test_full = read_mnist_images('mnist/t10k-images-idx3-ubyte.gz')
y_mnist_test_full = read_mnist_labels('mnist/t10k-labels-idx1-ubyte.gz')

# ─── THESIS-OPTIMIZED: subsample for faster runtime ────────────
TRAIN_SAMPLES = 30000     # v3: was 15000 — more data → stronger learning
TEST_SAMPLES  = 5000      # v3: was 3000  — more reliable accuracy estimates
# v7: CIFAR-100 has 100 classes on the same image budget → fewer samples/class,
# so it is intentionally harder. Kept at the same sample budget as CIFAR-10 for
# a fair FLOPs/communication comparison across datasets.
TRAIN_SAMPLES_C100 = 30000
TEST_SAMPLES_C100  = 5000

# Reproducible random subsample (uses SEED defined in step 4)
_rng_sub = np.random.default_rng(SEED)

def subsample(X, y, n, rng):
    idx = rng.permutation(len(X))[:n]
    return X[idx], y[idx]

X_cifar,      y_cifar      = subsample(X_cifar_full,      y_cifar_full,      TRAIN_SAMPLES, _rng_sub)
X_cifar_test, y_cifar_test = subsample(X_cifar_test_full, y_cifar_test_full, TEST_SAMPLES,  _rng_sub)
X_mnist,      y_mnist      = subsample(X_mnist_full,      y_mnist_full,      TRAIN_SAMPLES, _rng_sub)
X_mnist_test, y_mnist_test = subsample(X_mnist_test_full, y_mnist_test_full, TEST_SAMPLES,  _rng_sub)
X_cifar100,      y_cifar100      = subsample(X_cifar100_full,      y_cifar100_full,
                                              TRAIN_SAMPLES_C100, _rng_sub)
X_cifar100_test, y_cifar100_test = subsample(X_cifar100_test_full, y_cifar100_test_full,
                                              TEST_SAMPLES_C100,  _rng_sub)

print()
print(f'{"Dataset":<12} {"Train":>10} {"Test":>8} {"Classes":>8} {"Input Shape"}')
print('─'*55)
print(f'{"CIFAR-10":<12} {len(X_cifar):>10,} {len(X_cifar_test):>8,} {10:>8}  {X_cifar.shape[1:]}')
print(f'{"CIFAR-100":<12} {len(X_cifar100):>10,} {len(X_cifar100_test):>8,} {100:>8}  {X_cifar100.shape[1:]}')
print(f'{"MNIST":<12} {len(X_mnist):>10,} {len(X_mnist_test):>8,} {10:>8}  {X_mnist.shape[1:]}')
print()
print(f'Total training samples : {len(X_cifar)+len(X_mnist)+len(X_cifar100):,}  '
      f'(subsampled from {len(X_cifar_full)+len(X_mnist_full)+len(X_cifar100_full):,})')


📥 Downloading CIFAR-10…
📦 Extracting…
✅ CIFAR-10 ready.
📥 Downloading CIFAR-100…
📦 Extracting…
✅ CIFAR-100 ready.
📥 Downloading train-images-idx3-ubyte.gz…
📥 Downloading train-labels-idx1-ubyte.gz…
📥 Downloading t10k-images-idx3-ubyte.gz…
📥 Downloading t10k-labels-idx1-ubyte.gz…
✅ MNIST ready.

Dataset           Train     Test  Classes Input Shape
───────────────────────────────────────────────────────
CIFAR-10         30,000    5,000       10  (3, 32, 32)
CIFAR-100        30,000    5,000      100  (3, 32, 32)
MNIST            30,000    5,000       10  (784,)

Total training samples : 90,000  (subsampled from 160,000)
